In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

candidates = [Path('titanic.csv'), Path('LAB1') / 'titanic.csv', Path('LAB1') / 'Bai2_Trang15-17' / 'titanic.csv', Path('../titanic.csv'), Path('/content/titanic.csv'), Path('/mnt/data/titanic.csv')]
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    try:
        from google.colab import files
    except ImportError:
        files = None
    if files is not None:
        print('Chưa tìm thấy titanic.csv. Hãy chọn tệp CSV để tải lên Colab.')
        uploaded = files.upload()
        uploaded_path = next((Path(name) for name in uploaded if Path(name).name.lower() == 'titanic.csv'), None)
        if uploaded_path is not None and uploaded_path.is_file():
            data_path = uploaded_path
if data_path is None:
    raise FileNotFoundError(f'Không tìm thấy titanic.csv. Thư mục chạy hiện tại: {Path.cwd()}. Hãy tải CSV lên Colab hoặc đặt CSV trong thư mục này rồi chạy lại ô.')
df = pd.read_csv(data_path)
plot_data = df.dropna(subset=['sibsp', 'parch', 'survived']).copy()
plot_data['nguoi_di_cung'] = plot_data['sibsp'] + plot_data['parch']
plot_data['nhom_di_cung'] = plot_data['nguoi_di_cung'].clip(upper=6).astype(int).astype(str)
plot_data.loc[plot_data['nguoi_di_cung'] >= 6, 'nhom_di_cung'] = '6+'
order = ['0', '1', '2', '3', '4', '5', '6+']
summary = (plot_data.groupby('nhom_di_cung', observed=True)['survived']
           .agg(ty_le_song='mean', so_nguoi='size').reindex(order).dropna().reset_index())
print('Số hành khách và tỷ lệ sống theo số người đi cùng:')
display(summary)
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(summary['nhom_di_cung'], summary['ty_le_song'], color='#4c78a8')
ax.set_title('Tỷ lệ sống sót theo số người đi cùng')
ax.set_xlabel('Số người đi cùng (6 nghĩa là 6 trở lên)')
ax.set_ylabel('Tỷ lệ sống sót')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.25)
for bar, n in zip(bars, summary['so_nguoi']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f'n={n}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

print('Nhận xét: so sánh tỷ lệ sống giữa người đi một mình và người có người thân đi cùng; các nhóm rất nhỏ (đặc biệt 6+) có độ bất định cao hơn.')